## Практическое задание №3

В файле **WIPO-GII_2022.xlsx** содержаться данные по странам о показателях, используемых для расчета Global Innovation Index за 2022 год. 

### В файле следующие переменные:																																
- Country
- Business environment
- Business sophistication index
- Citable documents H-index
- Creative goods and services
- Creative goods exports, % total trade
- Creative outputs index
- Cultural and creative services exports, % total trade
- Domestic market scale, bn PPP
- Ecological sustainability
- Expenditure on education, % GDP
- Firms offering formal training, %
- General infrastructure
- Government effectiveness
- Government's online service
- Graduates in science and engineering, %
- Gross expenditure on R&D, % GDP
- High-tech exports, % total trade
- High-tech imports, % total trade
- High-tech manufacturing, %
- Human capital and research index
- Information and Communications Technology access
- Information and Communications Technology services exports, % total trade
- Information and Communications Technology services imports, % total trade
- Information and Communications Technology use
- Infrastructure index
- Intangible assets
- Joint venture/strategic alliance deals/bn PPP GDP
- Knowledge absorption
- Knowledge and technology outputs index
- Knowledge workers
- Knowledge-intensive employment, %
- Logistics performance
- Market sophistication index
- Mobile app creation/bn PPP GDP
- Online creativity
- PISA scales in reading, maths and science
- Political and operational stability
- Political environment
- Pupil-teacher ratio, secondary
- QS university ranking average score of top 3 universities
- Research and development (R&D)
- Scientific and technical articles/bn PPP GDP
- Software spending, % GDP
- Tertiary education
- Tertiary enrolment, % gross
- University-industry R&D collaboration
- Region
- Region2
- Euro area
- European Union
- OECD members
- Least developed countries
- Income
- High income

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import calinski_harabasz_score, silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

df1 = pd.read_excel('WIPO-GII_2022.xlsx')

Выполните, пожалуйста, задания.

**1. Построение признакового пространства и снижение размерности для кластеризации**

- Выберите не менее 10 числовых признаков из датасета, которые потенциально подходят для построения модели кластеризации. Обоснуйте их выбор (информативность, вариативность, отсутствие пропусков, предметный смысл).

- Проведите корреляционный анализ выбранных признаков. Сделайте вывод о наличии или отсутствии мультиколлинеарности.

- При наличии значимых корреляций выполните снижение размерности данных. Используйте метод главных компонент (PCA). Возможно, что имеет смысл выделить компоненты не на основе всех рассматриваемых признаков, а только на части изначально отобранных. Определите оптимальное количество компонент. Для каждой выделенной компоненты укажите процент объяснённой вариации; проанализируйте нагрузки; дайте компоненте осмысленное название и обоснуйте его.

- Сохраните полученные компоненты в датасете в виде новых переменных и используйте их для дальнейшей кластеризации.

- Сформируйте финальный список признаков по которым будет проведена кластеризация. Это могут быть только исходные признаки; только сформированные компоненты или же комбинация из исходных признаков и компонентов. Приведите аргументы в пользу того, что финальный список признаков может быть использован для кластеризации объектов наблюдения.

**!Если не удается выделить компоненты на имеющихся данных, то приведите расчеты и аргументы, подтверждающие это.**

In [2]:
numeric_features = df1.select_dtypes(include='number').filter(regex='_2022$').columns.tolist()
display(pd.Series(numeric_features, name='Количественные показатели').to_frame())

,Количественные показатели
0,Business environment_2022
1,Business sophistication index_2022
2,Citable documents H-index_2022
3,Creative goods and services_2022
4,"Creative goods exports, % total trade_2022"
5,Creative outputs index_2022
6,"Cultural and creative services exports, % tota..."
7,"Domestic market scale, bn PPP$_2022"
8,Ecological sustainability_2022
9,"Expenditure on education, % GDP_2022"


**Выбор 10 числовых признаков**

Для кластеризации выбраны показатели, характеризующие условия для инноваций, подготовку кадров, научную деятельность и использование технологий. В названиях столбцов Excel к каждому показателю добавлен суффикс `_2022`.

| Признак | Обоснование выбора |
| --- | --- |
| Government effectiveness | Характеризует эффективность государственного управления и институциональные условия для развития науки и бизнеса. |
| General infrastructure | Отражает инфраструктурные условия, необходимые для деятельности предприятий и внедрения технологий. |
| Information and Communications Technology access | Показывает доступность цифровой инфраструктуры, важной для распространения знаний и технологий. |
| Expenditure on education, % GDP | Характеризует финансирование образования и вложения в подготовку человеческого капитала. |
| Graduates in science and engineering, % | Отражает подготовку специалистов научных и инженерных направлений, необходимых для инноваций. |
| Gross expenditure on R&D, % GDP | Характеризует вложения в исследования и разработки. |
| Scientific and technical articles/bn PPP$ GDP | Отражает результативность научной деятельности с учётом масштаба экономики. |
| Knowledge-intensive employment, % | Характеризует распространённость занятости в деятельности, требующей высокой квалификации и знаний. |
| High-tech manufacturing, % | Показывает роль высокотехнологичного производства в экономике. |
| High-tech exports, % total trade | Характеризует участие страны в международной торговле высокотехнологичной продукцией. |

Выбраны количественные характеристики, а не идентификаторы или бинарные категории. `Country` сохраняется для определения стран, а регионы, группы дохода и флаги членства в объединениях — для последующего сравнения кластеров. Общие индексы `Human capital and research index`, `Infrastructure index` и `Knowledge and technology outputs index` не включены вместе с их составляющими, чтобы избежать дополнительного повторного учёта одних характеристик. Возможные корреляции между выбранными показателями будут проверены на следующем шаге.

In [3]:
features = [
    'Government effectiveness_2022',
    'General infrastructure_2022',
    'Information and Communications Technology access_2022',
    'Expenditure on education, % GDP_2022',
    'Graduates in science and engineering, %_2022',
    'Gross expenditure on R&D, % GDP_2022',
    'Scientific and technical articles/bn PPP$ GDP_2022',
    'Knowledge-intensive employment, %_2022',
    'High-tech manufacturing, %_2022',
    'High-tech exports, % total trade_2022',
]

X = df1[features].dropna()

feature_stats = X.agg(['count', 'nunique', 'min', 'max', 'std']).T
feature_stats['missing_before'] = df1[features].isna().sum()
display(feature_stats.round(2))
print(f'Рабочая выборка: {X.shape[0]} стран, {X.shape[1]} признаков.')
print(f'Исключено стран с пропусками: {len(df1) - len(X)} из {len(df1)}.')

,count,nunique,min,max,std,missing_before
Government effectiveness_2022,132.0,124.0,0.0,100.0,19.79,7
General infrastructure_2022,132.0,113.0,2.9,76.2,15.48,7
Information and Communications Technology access_2022,132.0,109.0,0.0,99.6,17.47,7
"Expenditure on education, % GDP_2022",132.0,114.0,0.0,100.0,18.98,7
"Graduates in science and engineering, %_2022",132.0,98.0,0.0,100.0,28.09,7
"Gross expenditure on R&D, % GDP_2022",132.0,99.0,0.0,100.0,19.49,7
Scientific and technical articles/bn PPP$ GDP_2022,132.0,121.0,0.0,100.0,23.70,7
"Knowledge-intensive employment, %_2022",132.0,117.0,0.0,100.0,25.83,7
"High-tech manufacturing, %_2022",132.0,102.0,0.0,100.0,25.19,7
"High-tech exports, % total trade_2022",132.0,115.0,0.0,100.0,25.98,7


Рабочая выборка: 132 стран, 10 признаков.
Исключено стран с пропусками: 7 из 139.


**Вывод по отбору признаков.** В исходном файле 139 стран. Для 7 стран — Puerto Rico, Cabo Verde, Malawi, Sierra Leone, Lebanon, Solomon Islands и Bolivia — отсутствуют все 46 количественных показателей за 2022 год. Эти строки исключены только из рабочей таблицы `X`; исходная таблица `df1` сохранена. В результате для каждого из 10 выбранных признаков имеется 132 наблюдения без пропусков. Значения не восстанавливались искусственно, нули не заменялись на пропуски.

Признаки обладают вариативностью: число уникальных значений составляет от 98 до 124, стандартное отклонение — от 15,48 до 28,09. Таким образом, среди них нет постоянных показателей, которые не позволяют различать страны. В таблице `count` — число наблюдений, `nunique` — число уникальных значений, `std` — стандартное отклонение, `missing_before` — число пропусков до формирования рабочей выборки.

Выбранные признаки охватывают несколько содержательных сторон инновационного развития. Это исходный набор для следующего этапа — корреляционного анализа; окончательное признаковое пространство будет определено после проверки связей и решения о необходимости PCA. Названия с `% GDP` и другими единицами сохранены из Excel; фактические значения выбранных показателей находятся в диапазоне от 0 до 100, поэтому их нельзя автоматически интерпретировать как исходные проценты или физические величины без уточнения шкалы источника.

**2. Кластерный анализ и интерпретация результатов**

- Выберите подходящий алгоритм кластеризации. Кратко обоснуйте выбор (тип данных, размер выборки, интерпретируемость, устойчивость к выбросам). Укажите, почему другие рассмотренные в рамках курса методы не были выбраны.

- Определите оптимальное количество кластеров. Представьте график(и). Обоснуйте принятое решение.

- Рассчитайте описательные статистики по каждому кластеру. Подробно опишите каждый кластер, выделите его ключевые характеристики, отличающие от других кластеров.

- Дайте каждому кластеру осмысленное название. Обоснуйте выбор названий. Сформулируйте итоговые выводы о полученной сегментации.

In [1]:
# здесь можно написать код

**3. Апостериорный анализ и размышления над результатами**

- Сравните кластеры на основе значений категориальных переменных датасета. Например, рассмотрите как распределены страны разных регионов по кластерам или в какие кластеры в основном попали страны-члены ОЭСР? Сделайте содержательные выводы.

- Почему сформировалась именно такая кластерная структура? Соответствуют ли результаты вашим ожиданиям или оказались неожиданными?

In [ ]:
# здесь можно написать ответ

### Комментарии к заданию и принципам его оценивания:
 
 - Все пункты задания должны сопровождаться _выполненным_ фрагментом кода.
 - Если результаты в виде статистических показателей, таблиц или графиков никак не прокомментированы и отсутствуют содержательные выводы на их основе, то работа оценивается в 0 баллов, в этом случае анализ данных не проведён.
 - Правильно выполненное задание оценивается в 8 баллов (отлично). Для получения более высокой оценки нужно выйти за рамки предложенного задания. В случае с текущим заданием это может быть дополнительное самостоятельное изучение других алгоритмов кластеризации и их применение для анализа данных, или развернутое и глубокое исследование имеющихся данных на основе применения всех пройденных на занятиях алгоритмов кластеризации, сравнением результатов и выбором лучшей кластерной модели с объяснением своего выбора.
 - В задании три пункта. Вес первого - 35%, второго - 50%, третьего - 15%. 
 - Задание можно выполнять индивидуально или в группе из 2 человек, критерии оценивания при этом остаются одинаковыми.
 - Если задание выполняется вдвоем, то в Smart LMS ноутбук с решениями загружает только один из студентов.
 - В названии загружаемого в Smart LMS файла нужно указать фамилию(ии) автора(ов) работы.
 - При использовании инструментов искусственного интеллекта необходимо отметить это в работе в соответствии с правилами НИУ ВШЭ.

### Этот файл нужно сохранить как Jupyter Notebook (.ipynb) и загрузить в Smart LMS до 21:00 9 ноября. Не рекомендую загружать файл в Smart LMS в последние несколько минут перед дедлайном. После дедлайна выполненные задания не принимаются.